# 3단계 — attention/KV 개입 (P1a·P2)

주 가설 **H3**: attention/KV 개입이 준수 선호 점수를 **인과적으로** 바꾼다.
2단계(상관 관측)에서 확인한 attention/KV를 **직접 치환·조정**해 준수 결정이 바뀌는지 본다.

관련 계획서 절: **3.5**(준수 선호 점수) · **3.6**(개입).

> **오픈 모델(Qwen)에서만 가능** — 내부 접근 필요. **양자화 금지**(activation 흔들림 → 노이즈).

진행 순서: **① 불변식 자기검증(게이트) → ② A분할 기저 gap → ③ B분할 개입 → ④ 집계**.

핵심 규칙(CLAUDE.md, 코드로 강제):
- 준수 선호 점수 값으로 표본 **선별 안 함**(A/B는 고정 seed 셔플).
- 후보 쌍은 **모든 조건 동일 고정 문자열**.
- 개입 pair **토큰 정확 일치**(정렬 assert).
- **P1a=KV group(4) 단위, P2=query head(28) 단위**(GQA 7:1).


## 0. GPU 확인
양자화 없이 fp16으로 돌린다. 3B fp16은 T4(16GB)에 올라간다.
**GPU가 안 뜨면** 런타임 → 런타임 유형 변경 → T4 GPU.


In [ ]:
!nvidia-smi -L


## 1. repo clone (main)
코드는 **main**에서 받는다. 이미 clone돼 있으면 main 최신으로 맞춘다.


In [ ]:
REPO_URL = "https://github.com/deanjs/instruction-adherence.git"
BRANCH = "main"
import os
if not os.path.isdir("instruction-adherence"):
    !git clone --branch {BRANCH} {REPO_URL}
%cd instruction-adherence
!git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -1


## 2. 의존성
Colab의 torch(CUDA 빌드)는 유지하고 transformers/accelerate만 맞춘다.
`AttentionInterface` 등록에 transformers>=4.51 필요.


In [ ]:
!pip install -q "transformers>=4.51.0" "accelerate>=0.26.0"
import torch, transformers
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())


## 3. 검증 (게이트) — 개입 하네스 불변식

실측 전에 하네스가 **정의대로 동작**하는지 확인한다. **PASS여야 개입으로 넘어간다.**
- V1 개입 없는 경로 == 표준 SDPA (비트 근사)
- V2 no-op(donor=self) → 준수 선호 점수 불변
- V3 지침 구간 patch → Δ≈0 (causal mask상 비트 동일)
- V4 **GQA 단위**: n_q=28·n_kv=4, P1a=KV group·P2=query head
- V5 P2 질량 보존(α 행합=1)  ·  V6 λ=1 항등

검증은 크기 무관 → 1.5B fp32로 tight tolerance.


In [ ]:
!python src/stage3_intervention.py --validate


## 4. A분할 — 기저 gap (준수 − 손상)

개입 없이 손상본(위반 prefix)·준수본(준수 prefix)의 준수 선호 점수만 잰다.
gap = 준수 − 손상 = **완전 회복 목표**. Recovery Ratio의 분모(개입 실행 B와 분리).
결과는 Drive의 `stage3_intervention.jsonl`에 append(재개 가능).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
OUT = "/content/drive/MyDrive/instruction-adherence/stage3_intervention.jsonl"
os.makedirs(os.path.dirname(OUT), exist_ok=True)
!python src/stage3_intervention.py --run-a --n-seeds 3 --out "{OUT}"


## 5. B분할 — 개입 실행 (P1a 주 · P2 보조 · 음성 대조)

- **P1a(주)**: 위반 prefix 코드 K/V를 준수 값으로 KV group 단위 치환 → α 재계산.
- **P2(보조)**: 지침 구간 α에 λ∈{0.5,1,2,4,8} 배율(질량 보존/비보존).
- **음성 대조**: no-op·지침 patch·무작위 단위 → 귀무분포.

`--sweep`을 붙이면 층×단위 국소화까지 돈다(수천 config, 재개 가능·prefix 캐시 재사용).
먼저 스윕 없이 주 결과만 확인 권장.


In [ ]:
!python src/stage3_intervention.py --run-b --p1a --p2 --controls \
    --n-seeds 3 --out "{OUT}"


**(선택) 층×단위 국소화 스윕** — 어느 층/그룹이 회복을 이끄는지.


In [ ]:
!python src/stage3_intervention.py --run-b --p1a --p2 --controls --sweep \
    --n-seeds 3 --out "{OUT}"


## 6. 집계 — Recovery Ratio · 귀무검정

config별 평균 Δ, Recovery Ratio(=평균Δ ÷ A분할 gap), 무작위 대조 대비 상위 꼬리 p.
**주 판정**: P1a(p1a_full)가 무작위 귀무분포 상위 꼬리인가.
**건전성**: no-op·지침 patch 대조는 Δ≈0이어야 한다.


In [ ]:
!python src/stage3_intervention.py --summary-only --out "{OUT}"


## 7. 결과 내려받기 (선택)
`stage3_intervention.jsonl`은 사전 등록 기록이다. Drive에 이미 있다.


In [ ]:
from google.colab import files
files.download(OUT)
